In [0]:
dbutils.fs.ls("abfss://ingestion@musedatapipeline.dfs.core.windows.net/")

In [0]:
from pyspark.sql import SparkSession
import logging as l
from pyspark.sql import functions as F

In [0]:
print("hello")

In [0]:
dbutils.fs.ls("abfss://ingestion@musedatapipeline.dfs.core.windows.net/ingestion/sample/2026/06/")

In [0]:
raw_path = "abfss://ingestion@musedatapipeline.dfs.core.windows.net/ingestion/sample//2026/06/14/*.json"

df_raw = spark.read \
    .option("multiline", "true") \
    .json(raw_path)

print(f"Raw file count: {df_raw.count()}")

In [0]:
df_raw = df_raw.withColumn("_ingestion_date", F.current_date())

In [0]:
df_raw.show()

In [0]:
results_df = df_raw.select(F.explode("results").alias("job"), "_ingestion_date")

In [0]:
bronze_df = results_df.select("job.*", "_ingestion_date")

In [0]:
bronze_df.show()

In [0]:
bronze_df.printSchema()

In [0]:
spark.sql("DROP TABLE IF EXISTS muse.bronze.job_muse_bronze")


In [0]:
spark.sql("SHOW TABLES IN muse.bronze").show()

In [0]:
bronze_df.write \
    .format("delta") \
    .mode("append") \
    .option("path", "abfss://bronze@musedatapipeline.dfs.core.windows.net/bronze/jobs") \
    .saveAsTable("muse.bronze.job_muse_bronze_landing")

In [0]:
spark.sql("DESCRIBE DETAIL muse.bronze.job_muse_bronze").select("location").show(truncate=False)

In [0]:
%sql
drop table if exists muse.bronze.table_job

In [0]:
%sql

          select * from 
          muse.bronze.job_muse_bronze
          
     

In [0]:
print("Bronze layer ingestion completed!!")